# SDCP Hyperparameter Ablation

**Goal**: Vary α (query weight), β (positive prior weight), γ (negative prior weight) and
show SDCP is robust across a range of settings.

Default: α=0.45, β=0.35, γ=0.20

**Run on TruthfulQA only** (615Q, fastest) to keep runtime manageable.
Kernel must have model loaded (run Cell 0 only if fresh kernel).

In [1]:
# ── Cell 0: Fresh-kernel setup (skip if model already loaded) ─────────────────
import os, sys, gc, time, json
import numpy as np, pandas as pd
import torch, faiss, psutil
from datasets import load_from_disk
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer
from rouge_score import rouge_scorer as rs_module
from sklearn.metrics.pairwise import cosine_similarity
from datetime import datetime
from tqdm import tqdm

MODELS_DIR   = '/home/user/RAG_Best_Practices/models'
DATASETS_DIR = '/home/user/RAG_Best_Practices/datasets'
OUTPUT_DIR   = '/home/user/RAG_Best_Practices/outputs'

N_TRUTHFULQA = 615
RANDOM_SEED  = 42
INST_S = '[INST]'
INST_E = '[/INST]'
SYS    = 'You are a truthful expert question-answering bot and should correctly and concisely answer the following question'

def show_mem():
    ram  = psutil.virtual_memory()
    vram = torch.cuda.memory_allocated()/1024**3 if torch.cuda.is_available() else 0
    print(f'  RAM {ram.used/1024**3:.1f}/{ram.total/1024**3:.1f}GB | VRAM {vram:.1f}GB')

gc.collect(); torch.cuda.empty_cache(); show_mem()
MODEL_PATH = f'{MODELS_DIR}/mistral-7b'
EMBED_PATH = f'{MODELS_DIR}/minilm'
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
llm = AutoModelForCausalLM.from_pretrained(MODEL_PATH, quantization_config=bnb, device_map='auto')
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, padding_side='left')
tokenizer.pad_token = tokenizer.eos_token
embed_model = SentenceTransformer(EMBED_PATH).to(DEVICE)
show_mem()
print('Models ready!')

  RAM 15.8/47.0GB | VRAM 0.0GB


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

  RAM 16.6/47.0GB | VRAM 4.3GB
Models ready!


In [2]:
# ── Cell 1: Load TruthfulQA + build KB index ──────────────────────────────────
print('Loading TruthfulQA...')
tqa_raw = load_from_disk(f'{DATASETS_DIR}/truthfulqa').to_pandas()
tqa_all = tqa_raw[['question','best_answer','correct_answers','incorrect_answers']].copy()
tqa_all['correct_answers']   = tqa_all['correct_answers'].apply(lambda x: x.tolist() if isinstance(x, np.ndarray) else [x])
tqa_all['incorrect_answers'] = tqa_all['incorrect_answers'].apply(lambda x: x.tolist() if isinstance(x, np.ndarray) else [x])
tqa_all['best_answer']       = tqa_all['best_answer'].apply(lambda x: [x] if x else [])
tqa_all = tqa_all[(tqa_all['correct_answers'].apply(len) > 1) &
                   (tqa_all['incorrect_answers'].apply(len) > 1)].reset_index(drop=True)
tqa_test_idx = tqa_all.sample(n=N_TRUTHFULQA, random_state=RANDOM_SEED).index
tqa    = tqa_all.loc[tqa_test_idx].reset_index(drop=True)
tqa_kb = tqa_all.drop(tqa_test_idx).reset_index(drop=True)
del tqa_raw; gc.collect()
print(f'  test={len(tqa)}Q | KB={len(tqa_kb)}Q')

# Build FAISS index (same-dataset KB — same as SDCP-v2 official run)
print('Building FAISS index...')
kb_embs = embed_model.encode(tqa_kb['question'].tolist(), show_progress_bar=True, batch_size=64)
kb_embs = np.array(kb_embs, dtype=np.float32)
faiss.normalize_L2(kb_embs)
faiss_idx = faiss.IndexFlatIP(kb_embs.shape[1])
faiss_idx.add(kb_embs)
print(f'  FAISS: {faiss_idx.ntotal} vectors')
print('Ready!')

Loading TruthfulQA...
  test=615Q | KB=100Q
Building FAISS index...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

  FAISS: 100 vectors
Ready!


In [3]:
# ── Cell 2: Core inference functions ─────────────────────────────────────────
def generate(prompts, max_new_tokens=25, num_beams=2):
    enc = tokenizer(prompts, return_tensors='pt', padding=True,
                    truncation=True, max_length=2048).to(DEVICE)
    input_len = enc['input_ids'].shape[1]
    with torch.no_grad():
        out = llm.generate(input_ids=enc['input_ids'],
                           attention_mask=enc['attention_mask'],
                           max_new_tokens=max_new_tokens, num_beams=num_beams,
                           pad_token_id=tokenizer.eos_token_id)
    return [tokenizer.decode(r[input_len:], skip_special_tokens=True).strip()
            or 'I have no comment' for r in out]

def get_token_probs(prompt, max_new_tokens=20):
    enc = tokenizer(prompt, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        out = llm.generate(**enc, max_new_tokens=max_new_tokens,
                           return_dict_in_generate=True, output_scores=True,
                           pad_token_id=tokenizer.eos_token_id)
    text = tokenizer.decode(out.sequences[0][enc['input_ids'].shape[1]:],
                             skip_special_tokens=True).strip()
    cert = 0.0
    if out.scores:
        probs = torch.softmax(out.scores[0][0], dim=-1)
        top2  = torch.topk(probs, 2).values
        cert  = (top2[0] - top2[1]).item()
    return text, cert

def clean_response(resp):
    for stop in ['\nQuestion:','\nQ:','\n---','\nIncorrect','\nCorrect',
                 '\nVERIFIED','\nExample','\n\n','\nMy initial','\nContext:',
                 '\nFor the question','\nCommon']:
        if stop in resp: resp = resp[:resp.index(stop)]
    return resp.strip().strip('"').strip("'") or 'I have no comment'

def run_sdcp_with_params(test_data, kb_data, fi, alpha, beta, gamma,
                          top_k=15, top_k_ctx=4, label=''):
    """Run SDCP-v2 (always uncertain) with given α, β, γ."""
    scorer   = rs_module.RougeScorer(['rouge1'], use_stemmer=True)
    r1s, ecss = [], []

    for idx, row in tqdm(test_data.iterrows(), total=len(test_data),
                          desc=f'α={alpha} β={beta} γ={gamma}', leave=False):
        query       = row['question']
        best_answer = row['best_answer'] if isinstance(row['best_answer'], list) else [row['best_answer']]

        # Priors
        pos_prompt = f'{INST_S}{SYS}\nQuestion: {query}\nAnswer concisely:{INST_E}'
        p_pos, _   = get_token_probs(pos_prompt, max_new_tokens=20)
        p_pos      = clean_response(p_pos)
        neg_prompt = (f'{INST_S}What is a common misconception or false belief '
                      f'that people hold about this topic?\n'
                      f'Question: {query}\nCommon wrong belief (very short):{INST_E}')
        p_neg = clean_response(generate([neg_prompt], max_new_tokens=20)[0])

        # Retrieval with contrastive scoring
        q_emb   = np.array(embed_model.encode([query],  show_progress_bar=False), dtype=np.float32)
        pos_emb = np.array(embed_model.encode([p_pos],  show_progress_bar=False), dtype=np.float32)
        neg_emb = np.array(embed_model.encode([p_neg],  show_progress_bar=False), dtype=np.float32)
        faiss.normalize_L2(q_emb)
        _, idxs = fi.search(q_emb, top_k)

        sentences = []
        for i in idxs[0]:
            if i < len(kb_data):
                r = kb_data.iloc[i]
                sentences.append(r['question'])
                ba = r['best_answer']
                ba_t = ba[0] if isinstance(ba, list) and ba else str(ba)
                if ba_t and len(ba_t) > 3: sentences.append(ba_t)

        sentences = [s for s in sentences if s and len(s.strip()) > 5]
        if sentences:
            s_embs = embed_model.encode(sentences, show_progress_bar=False)
            q_sims = cosine_similarity(s_embs, q_emb).flatten()
            p_sims = cosine_similarity(s_embs, pos_emb).flatten()
            n_sims = cosine_similarity(s_embs, neg_emb).flatten()
            scores = alpha * q_sims + beta * p_sims - gamma * n_sims
            context = ' '.join([sentences[i] for i in np.argsort(-scores)[:top_k_ctx]])

            kb_ex = kb_data.iloc[idxs[0][0]]
            kb_correct   = (kb_ex['best_answer'][0] if isinstance(kb_ex['best_answer'], list)
                            and kb_ex['best_answer'] else str(kb_ex['best_answer']))
            kb_incorrect = (kb_ex['incorrect_answers'][0]
                            if isinstance(kb_ex['incorrect_answers'], list)
                            and kb_ex['incorrect_answers'] else '')
            kb_q = kb_ex['question']

            prompt = (f'{INST_S}{SYS}\nRetrieved context: {context}\n'
                      f'Example — Q: {kb_q}\n'
                      f'  Correct: {kb_correct}\n'
                      f'  Incorrect: {kb_incorrect}\n'
                      f'My initial thought: {p_pos}\n'
                      f'Common mistake to avoid: {p_neg}\n'
                      f'Question: {query}\nVerified answer:{INST_E}')
        else:
            prompt = f'{INST_S}{SYS}\nQuestion: {query}\nAnswer:{INST_E}'

        final = clean_response(generate([prompt], max_new_tokens=25)[0])

        best_r1 = max((scorer.score(ref, final)['rouge1'].fmeasure
                       for ref in best_answer if ref), default=0)
        r1s.append(best_r1 * 100)
        try:
            embs = embed_model.encode([best_answer[0], final])
            ecss.append(float(cosine_similarity([embs[0]], [embs[1]])[0][0]) * 100)
        except:
            ecss.append(0.0)

        if idx % 30 == 0: gc.collect(); torch.cuda.empty_cache()

    r1  = float(np.mean(r1s))
    ecs = float(np.mean(ecss))
    print(f'  [{label}] α={alpha:.2f} β={beta:.2f} γ={gamma:.2f}  →  R1={r1:.2f}  ECS={ecs:.2f}')
    return {'alpha': alpha, 'beta': beta, 'gamma': gamma, 'R1': r1, 'ECS': ecs, 'label': label}

print('Functions ready.')

Functions ready.


In [4]:
# ── Cell 3: Vary α (query similarity weight) — β and γ fixed ─────────────────
# Default: α=0.45, β=0.35, γ=0.20
# Vary α: 0.20, 0.30, 0.45*, 0.55, 0.70
print('=== Varying α (query weight) | β=0.35 γ=0.20 fixed ===')
results_alpha = []
for alpha in [0.20, 0.30, 0.45, 0.55, 0.70]:
    beta  = 0.35
    gamma = 0.20
    t0 = time.time()
    res = run_sdcp_with_params(tqa, tqa_kb, faiss_idx, alpha, beta, gamma,
                                label=f'alpha={alpha}')
    res['elapsed_min'] = (time.time() - t0) / 60
    results_alpha.append(res)

print('\nα sweep done:')
for r in results_alpha:
    star = ' ← default' if r['alpha'] == 0.45 else ''
    print(f"  α={r['alpha']:.2f}  R1={r['R1']:.2f}  ECS={r['ECS']:.2f}{star}")

=== Varying α (query weight) | β=0.35 γ=0.20 fixed ===


α=0.2 β=0.35 γ=0.2:   0%|          | 0/615 [00:00<?, ?it/s]From v4.47 onwards, when a model cache is to be returned, `generate` will return a `Cache` instance instead by default (as opposed to the legacy tuple of tuples format). If you want to keep returning the legacy format, please set `return_legacy_cache=True`.
                                                                     

  [alpha=0.2] α=0.20 β=0.35 γ=0.20  →  R1=35.65  ECS=65.41


  [alpha=0.3] α=0.30 β=0.35 γ=0.20  →  R1=35.90  ECS=66.08


  [alpha=0.45] α=0.45 β=0.35 γ=0.20  →  R1=35.79  ECS=65.62


  [alpha=0.55] α=0.55 β=0.35 γ=0.20  →  R1=35.57  ECS=65.25


  [alpha=0.7] α=0.70 β=0.35 γ=0.20  →  R1=35.81  ECS=65.36

α sweep done:
  α=0.20  R1=35.65  ECS=65.41
  α=0.30  R1=35.90  ECS=66.08
  α=0.45  R1=35.79  ECS=65.62 ← default
  α=0.55  R1=35.57  ECS=65.25
  α=0.70  R1=35.81  ECS=65.36


In [5]:
# ── Cell 4: Vary β (positive prior weight) ────────────────────────────────────
print('=== Varying β (positive prior weight) | α=0.45 γ=0.20 fixed ===')
results_beta = []
for beta in [0.10, 0.20, 0.35, 0.50, 0.65]:
    alpha = 0.45
    gamma = 0.20
    t0 = time.time()
    res = run_sdcp_with_params(tqa, tqa_kb, faiss_idx, alpha, beta, gamma,
                                label=f'beta={beta}')
    res['elapsed_min'] = (time.time() - t0) / 60
    results_beta.append(res)

print('\nβ sweep done:')
for r in results_beta:
    star = ' ← default' if r['beta'] == 0.35 else ''
    print(f"  β={r['beta']:.2f}  R1={r['R1']:.2f}  ECS={r['ECS']:.2f}{star}")

=== Varying β (positive prior weight) | α=0.45 γ=0.20 fixed ===


  [beta=0.1] α=0.45 β=0.10 γ=0.20  →  R1=35.88  ECS=65.75


  [beta=0.2] α=0.45 β=0.20 γ=0.20  →  R1=36.21  ECS=65.95


  [beta=0.35] α=0.45 β=0.35 γ=0.20  →  R1=35.79  ECS=65.62


  [beta=0.5] α=0.45 β=0.50 γ=0.20  →  R1=35.70  ECS=65.73


  [beta=0.65] α=0.45 β=0.65 γ=0.20  →  R1=35.57  ECS=65.70

β sweep done:
  β=0.10  R1=35.88  ECS=65.75
  β=0.20  R1=36.21  ECS=65.95
  β=0.35  R1=35.79  ECS=65.62 ← default
  β=0.50  R1=35.70  ECS=65.73
  β=0.65  R1=35.57  ECS=65.70


In [6]:
# ── Cell 5: Vary γ (negative prior weight) ────────────────────────────────────
print('=== Varying γ (negative prior weight) | α=0.45 β=0.35 fixed ===')
results_gamma = []
for gamma in [0.00, 0.10, 0.20, 0.35, 0.50]:
    alpha = 0.45
    beta  = 0.35
    t0 = time.time()
    res = run_sdcp_with_params(tqa, tqa_kb, faiss_idx, alpha, beta, gamma,
                                label=f'gamma={gamma}')
    res['elapsed_min'] = (time.time() - t0) / 60
    results_gamma.append(res)

print('\nγ sweep done:')
for r in results_gamma:
    star = ' ← default' if r['gamma'] == 0.20 else ''
    note = ' (= no neg prior)' if r['gamma'] == 0.00 else ''
    print(f"  γ={r['gamma']:.2f}  R1={r['R1']:.2f}  ECS={r['ECS']:.2f}{star}{note}")

=== Varying γ (negative prior weight) | α=0.45 β=0.35 fixed ===


  [gamma=0.0] α=0.45 β=0.35 γ=0.00  →  R1=35.27  ECS=64.68


  [gamma=0.1] α=0.45 β=0.35 γ=0.10  →  R1=35.45  ECS=65.02


  [gamma=0.2] α=0.45 β=0.35 γ=0.20  →  R1=35.79  ECS=65.62


  [gamma=0.35] α=0.45 β=0.35 γ=0.35  →  R1=35.83  ECS=65.40


  [gamma=0.5] α=0.45 β=0.35 γ=0.50  →  R1=35.19  ECS=64.69

γ sweep done:
  γ=0.00  R1=35.27  ECS=64.68 (= no neg prior)
  γ=0.10  R1=35.45  ECS=65.02
  γ=0.20  R1=35.79  ECS=65.62 ← default
  γ=0.35  R1=35.83  ECS=65.40
  γ=0.50  R1=35.19  ECS=64.69


In [7]:
# ── Cell 6: Save all results ──────────────────────────────────────────────────
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
all_results = {
    'alpha_sweep': results_alpha,
    'beta_sweep':  results_beta,
    'gamma_sweep': results_gamma,
}
out_path = f'{OUTPUT_DIR}/sdcp_hyperparam_ablation_{ts}.json'
with open(out_path, 'w') as f:
    json.dump(all_results, f, indent=2)
print(f'Saved to: {out_path}')

# Print LaTeX-ready table rows
print('\n=== LaTeX Table Rows ===')
print('\\midrule')
print('% α sweep')
for r in results_alpha:
    star = '$^\\star$' if r['alpha'] == 0.45 else ''
    print(f"$\\alpha$={r['alpha']:.2f}{star} & {r['R1']:.2f} & {r['ECS']:.2f} \\\\")
print('\\midrule')
print('% β sweep')
for r in results_beta:
    star = '$^\\star$' if r['beta'] == 0.35 else ''
    print(f"$\\beta$={r['beta']:.2f}{star} & {r['R1']:.2f} & {r['ECS']:.2f} \\\\")
print('\\midrule')
print('% γ sweep')
for r in results_gamma:
    star = '$^\\star$' if r['gamma'] == 0.20 else ''
    print(f"$\\gamma$={r['gamma']:.2f}{star} & {r['R1']:.2f} & {r['ECS']:.2f} \\\\")

Saved to: /home/user/RAG_Best_Practices/outputs/sdcp_hyperparam_ablation_20260507_065602.json

=== LaTeX Table Rows ===
\midrule
% α sweep
$\alpha$=0.20 & 35.65 & 65.41 \\
$\alpha$=0.30 & 35.90 & 66.08 \\
$\alpha$=0.45$^\star$ & 35.79 & 65.62 \\
$\alpha$=0.55 & 35.57 & 65.25 \\
$\alpha$=0.70 & 35.81 & 65.36 \\
\midrule
% β sweep
$\beta$=0.10 & 35.88 & 65.75 \\
$\beta$=0.20 & 36.21 & 65.95 \\
$\beta$=0.35$^\star$ & 35.79 & 65.62 \\
$\beta$=0.50 & 35.70 & 65.73 \\
$\beta$=0.65 & 35.57 & 65.70 \\
\midrule
% γ sweep
$\gamma$=0.00 & 35.27 & 64.68 \\
$\gamma$=0.10 & 35.45 & 65.02 \\
$\gamma$=0.20$^\star$ & 35.79 & 65.62 \\
$\gamma$=0.35 & 35.83 & 65.40 \\
$\gamma$=0.50 & 35.19 & 64.69 \\
